# Neural Networks

## Introduction

This notebook continues the modeling workflow for predicting customer churn by introducing a neural network implemented in PyTorch. This represents a shift from classical machine learning models toward a more flexible function approximation framework based on layered transformations.

In earlier stages, logistic regression provided an interpretable linear baseline, while tree-based methods (decision trees, random forests, and gradient boosting) captured non-linear relationships through recursive partitioning of the feature space. Support Vector Machines further extended this exploration by introducing margin-based learning and kernel methods, enabling controlled non-linearity without explicit feature engineering.

Neural networks offer a different perspective by learning hierarchical representations of the input data. Instead of relying on predefined transformations or decision rules, the model learns a sequence of transformations that map input features to the target variable. This allows for flexible modeling of complex relationships, even in relatively structured tabular datasets such as customer churn data.

In this notebook, the focus is intentionally kept on a simple feed-forward neural network with a single hidden layer. This design choice serves two purposes. First, it provides a natural extension of logistic regression: a neural network with zero hidden layers is equivalent to logistic regression, while adding one hidden layer introduces a controlled form of non-linearity. Second, it avoids unnecessary architectural complexity, allowing us to isolate the impact of introducing learned non-linear feature interactions.

While deeper architectures are common in domains such as computer vision and natural language processing, their benefits in structured tabular data are often limited. In many practical cases, shallow networks perform comparably to deeper ones, especially when feature engineering and data preprocessing are already well-developed. As a result, increasing depth does not guarantee improved performance and may introduce additional challenges such as overfitting and training instability.

Unlike tree-based models, neural networks are sensitive to the scale of input features. Therefore, feature standardization remains a critical component of the modeling pipeline to ensure stable and efficient optimization.

From an interpretability standpoint, neural networks are less transparent than linear models and do not provide explicit decision rules like trees. Instead, they operate as parameterized function approximators, where predictions emerge from learned weights distributed across the network. While this limits direct interpretability, it enables the model to capture interactions and non-linear effects that may be difficult to specify manually.

The objective of this stage is to evaluate whether a simple neural network can improve predictive performance relative to previously explored models, particularly in terms of probability estimation and ranking ability. As in prior notebooks, the evaluation framework remains consistent: model selection is performed using cross-validation on the training data, a validation set is used for comparison and threshold tuning, and a final hold-out test set is reserved for unbiased performance assessment.

Performance is assessed using the same set of metrics as before, including ROC-AUC, PR-AUC, and threshold-dependent measures. This ensures a fair and consistent comparison across linear, tree-based, kernel-based, and neural network approaches.

The results of this notebook will provide a baseline neural network benchmark and complete the comparison across the major modeling paradigms considered in this project.

## Table of Contents

1. [Modeling Considerations](#modeling-considerations)
2. [Model Definition and Hyperparameter Tuning](#model-definition-and-hyperparameter-tuning)
3. [Data Preparation](#data-loading-and-preparation)
4. [Batch Size 16](#batch-size--16)
5. [Batch Size 32](#batch-size--32)
6. [Batch Size 64](#batch-size--64)
7. [Test Set Evaluation](#test-set-evaluation)
5. [Executive Summary](#executive-summary)

## Modeling Considerations

Before fitting the neural network model, we outline several key considerations that influence model behavior, training stability, and evaluation.

---

### Class Imbalance

The target variable exhibits moderate class imbalance, with approximately 26.5% of customers churning.

Neural networks trained with standard loss functions (such as binary cross-entropy) can be sensitive to class imbalance, as the optimization process may favor the majority class to minimize overall loss. This can lead to high accuracy but poor performance in identifying churners.

To address this, accuracy is not used as a primary evaluation metric. Instead, we focus on recall, precision, F1 score, and PR-AUC, which better reflect performance on the minority class. Additionally, class imbalance can be handled directly within the loss function by applying class weights, increasing the penalty for misclassifying churners.

---

### Feature Scaling

Feature scaling is essential for neural networks.

Unlike tree-based models, neural networks rely on gradient-based optimization. Features with different scales can lead to unstable gradients, slow convergence, and suboptimal solutions.

To ensure stable and efficient training, all numerical features are standardized prior to model fitting. This allows the optimization process to proceed smoothly and ensures that no single feature disproportionately influences the learning dynamics.

---

### Feature Representation

Neural networks operate on numerical input and do not inherently handle categorical variables.

Categorical features are therefore encoded using one-hot encoding, while numerical features are retained in their continuous form. This representation is consistent with previous linear and SVM models, ensuring comparability across modeling approaches.

Unlike linear models, neural networks can learn non-linear interactions between features implicitly through the hidden layer. This reduces the need for manual feature engineering of interaction terms, as the network can approximate these relationships during training.

---

### Model Architecture

The neural network used in this notebook is a simple feed-forward architecture with a single hidden layer.

This design is intentionally chosen as a controlled extension of logistic regression. A model without hidden layers is equivalent to logistic regression, while introducing one hidden layer enables the model to capture non-linear relationships between features.

The number of neurons in the hidden layer controls the model's capacity. A small number of neurons results in a more constrained model, while a larger number increases flexibility but also the risk of overfitting.

Deeper architectures are not considered in this stage. For structured tabular data, increasing the number of layers often yields limited performance gains while significantly increasing model complexity, tuning effort, and training instability.

---

### Regularization and Optimization

Neural networks introduce several sources of model complexity that must be carefully controlled.

Regularization techniques such as weight decay (L2 regularization) and early stopping are used to prevent overfitting. Early stopping monitors performance on the validation set and halts training when no further improvement is observed.

Optimization is performed using gradient-based methods (e.g., Adam optimizer), which require tuning of hyperparameters such as learning rate and batch size. These parameters influence both convergence speed and final model performance.

Proper tuning is essential to ensure stable training and good generalization.

---

### Probability Estimation

Neural networks naturally produce probability estimates through the use of a sigmoid activation function in the output layer.

These probabilities represent the model’s estimated likelihood of churn and can be used directly for evaluation with ROC-AUC and PR-AUC. In this notebook, the analysis focuses primarily on probability-based evaluation rather than extensive threshold optimization, as threshold selection has already been explored in earlier models.

---

### Validation Strategy and Iterative Model Refinement

As with previous models, hyperparameter tuning and model refinement are performed iteratively, which introduces a risk of overfitting to evaluation data if not carefully controlled.

To mitigate this, the training data is split into a training subset and a separate validation set. Model parameters are learned on the training subset, while the validation set is used to monitor performance, tune hyperparameters, and guide early stopping.

A final hold-out test set remains reserved for unbiased performance evaluation and is not used during model development.

---

### Summary

Neural networks introduce a flexible, non-linear modeling approach that extends beyond the capabilities of linear models while differing fundamentally from tree-based methods. Their performance depends critically on proper feature scaling, controlled model capacity, and careful optimization.

By focusing on a simple architecture and maintaining a consistent evaluation framework, this stage aims to isolate the impact of learned non-linear feature interactions while avoiding unnecessary complexity.

With these considerations in place, we proceed to defining and training the neural network model.

## Model Definition and Hyperparameter Tuning

The neural network model is implemented in PyTorch as a feed-forward binary classifier. Unlike logistic regression, which estimates a single linear decision function, or tree-based methods, which partition the feature space through recursive splits, a neural network learns a sequence of transformations that map the input features to a predicted churn probability.

To maintain a structured and interpretable modeling workflow, the analysis is intentionally kept simple. The model is defined as a shallow neural network with a single hidden layer. This provides a natural extension of logistic regression: a network with no hidden layer is equivalent to logistic regression, while the introduction of one hidden layer allows the model to learn non-linear relationships between predictors without introducing unnecessary architectural complexity.

The architecture is therefore governed primarily by the size of the hidden layer. A smaller number of hidden units produces a more constrained model with limited capacity, while a larger number increases flexibility and the ability to capture more complex patterns. Because the dataset is structured tabular data rather than high-dimensional unstructured data, the goal is not to build a deep architecture but to assess whether a modest increase in model flexibility improves predictive performance.

In addition to the hidden layer size, several optimization-related hyperparameters influence model behavior. The learning rate controls the step size of the gradient updates and therefore affects both convergence speed and training stability. If the learning rate is too large, the optimization process may become unstable; if it is too small, training may converge slowly or become trapped in suboptimal solutions.

Batch size is another important consideration. The batch size determines how many observations are used to compute a single update of the model during training.

Instead of updating the model using the entire dataset at once, the data is divided into smaller subsets called batches. For each batch, the model makes predictions, computes the error, and updates its parameters. This process is repeated across all batches in multiple passes through the data (epochs).

Smaller batch sizes result in more frequent updates based on fewer observations. This introduces randomness into the optimization process, as each update is based on a slightly different subset of the data. While this can make training noisier, it can also help the model generalize better by preventing it from fitting too closely to the training data.

Larger batch sizes, on the other hand, produce more stable and consistent updates because each update is based on more data. This can lead to smoother training and faster computation on modern hardware, but may reduce the beneficial randomness that helps prevent overfitting.

The choice of batch size therefore represents a trade-off between stability and stochasticity in the optimization process.

Model complexity is further controlled through regularization. In this notebook, regularization is introduced through weight decay, which penalizes excessively large parameter values and helps reduce overfitting. In addition, early stopping is used as a practical form of regularization by halting training once validation performance no longer improves.

An additional form of regularization considered in this model is dropout. Dropout randomly deactivates a fraction of neurons during training, preventing the network from relying too heavily on any single pathway and encouraging more robust feature representations. While dropout is widely used in deep learning, its benefits in shallow networks for tabular data are often modest. Nevertheless, it provides a useful mechanism for evaluating whether additional stochastic regularization improves generalization in this setting.

Hyperparameter tuning is performed using a structured search over a limited set of candidate values. Rather than exploring a large and highly complex search space, the tuning process focuses on a small number of meaningful choices: hidden layer size, learning rate, batch size, weight decay, and dropout rate. This keeps the modeling process interpretable and aligned with the broader goal of introducing neural networks as a controlled extension of earlier models rather than as a highly engineered deep learning system.

Given the class imbalance in the dataset, model comparison is based primarily on PR-AUC, which emphasizes the model's ability to identify and rank churners correctly. This metric is more informative than accuracy in the current setting and complements ROC-AUC by focusing more directly on performance for the minority class.

Unlike classical machine learning models, neural networks are trained iteratively and rely on validation data during training. Therefore, a dedicated validation set is used both for early stopping and model selection, removing the need for cross-validation in this context.

This tuning strategy allows us to assess whether a simple neural network can provide meaningful gains over earlier models while preserving a disciplined and transparent validation framework. In the following sections, we define the neural network architecture, specify the candidate hyperparameter values, and train the model using this controlled search procedure.

## Data Loading and Preparation

The data loading and preprocessing steps used for the neural network closely follow the approach established in the logistic regression and Support Vector Machine notebooks.

Because neural networks require fully numerical inputs, categorical variables are represented using one-hot encoding, while numerical variables are retained in continuous form. As with the SVM model, feature scaling is applied to ensure stable optimization during training.

To maintain comparability across models, the same feature set and the same train, validation, and test split strategy are retained. This ensures that any differences in model performance can be attributed primarily to the modeling approach rather than changes in data preparation.

The prepared feature matrices are then converted into PyTorch tensors, which serve as inputs to the neural network.

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split

# Load datasets
train_df = pl.read_csv("./data/processed/06_dpp_train_df.csv")
test_df = pl.read_csv("./data/processed/06_dpp_test_df.csv")

selected_cols = [
    'SeniorCitizenRelevel',
    'Partner',
    'Dependents',
    'tenure',
    'Contract',
    'PaperlessBilling',
    'MonthlyCharges',
    'Churn',
    'AdditionalInternetServicesCount',
    'StreamingServicesCount',
    'PaymentMethod_bin_3'
]

train_df = train_df.select(selected_cols)
test_df = test_df.select(selected_cols)

# Convert to pandas for sklearn
train_pd = train_df.to_pandas()

# Stratified split: 25% of current train -> validation
train_sub_pd, val_pd = train_test_split(
    train_pd,
    test_size=0.25,
    stratify=train_pd["Churn"],
    random_state=42
)

# Back to polars
train_sub_df = pl.from_pandas(train_sub_pd)
val_df = pl.from_pandas(val_pd)

# Shapes
print("Train subset shape:", train_sub_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Class proportions helper
def class_proportions(df, name):
    print(f"\n{name} class proportions:")
    print(
        df.group_by("Churn")
        .len()
        .with_columns(
            (pl.col("len") / pl.col("len").sum()).alias("proportion")
        )
        .sort("Churn")
    )

class_proportions(train_sub_df, "Train subset")
class_proportions(val_df, "Validation")
class_proportions(test_df, "Test")

Train subset shape: (4225, 11)
Validation shape: (1409, 11)
Test shape: (1409, 11)

Train subset class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 3104 ┆ 0.734675   │
│ Yes   ┆ 1121 ┆ 0.265325   │
└───────┴──────┴────────────┘

Validation class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘

Test class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘


In [2]:
categorical_cols = [
    "SeniorCitizenRelevel", "Partner", "Dependents", "Contract", "PaperlessBilling", "PaymentMethod_bin_3"
]

numerical_cols = [
    "tenure", "MonthlyCharges", "AdditionalInternetServicesCount", "StreamingServicesCount"
]

target_col = "Churn"

# Define the order of categories for categorical variables, first will be dropped when one hot encoding
categorical_orders = {
    "SeniorCitizenRelevel": ["No", "Yes"],
    "Partner": ["No", "Yes"],
    "Dependents": ["No", "Yes"],
    "PaperlessBilling": ["No", "Yes"],
    "Contract": ["Month-to-month", "One year", "Two year"]
}

# check if we are not forgetting any column
set(categorical_cols + numerical_cols + [target_col]) == set(train_df.columns)

True

In [10]:
from src.utils.data_preparation_models import prepare_linear_features

train_X, train_y = prepare_linear_features(
    df=train_sub_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False
)

validation_X, validation_y = prepare_linear_features(
    df=val_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False,
    reference_columns= train_X.columns.tolist()
)

### Feature Scaling

At this stage, the training and validation datasets have been constructed using the shared data preparation pipeline, resulting in aligned feature matrices for both subsets.

Before training the neural network, an additional preprocessing step is required. Unlike tree-based methods, neural networks are sensitive to the scale of input features, as training relies on gradient-based optimization.

To ensure stable and efficient learning, features are standardized using z-score scaling. Each feature is transformed by subtracting its mean and dividing by its standard deviation, resulting in variables with approximately zero mean and unit variance.

The scaling parameters are estimated using the training data and then applied consistently to the validation and test sets. This approach preserves consistency across datasets while preventing data leakage, as information from the validation or test sets is not used during the fitting of the scaler.

Proper scaling helps ensure that gradients are well-behaved during optimization, leading to faster convergence and more stable training dynamics.

With scaling in place, the feature matrices are ready to be converted into tensors and used for neural network training.

In [11]:
from src.utils.data_preparation_models import fit_scaler, transform_with_scaler


scaler = fit_scaler(train_X)

scaled_train_X = transform_with_scaler(train_X, scaler)
scaled_validation_X = transform_with_scaler(validation_X, scaler)

Feature standardization is preferred over min-max scaling in this context, as it produces zero-centered inputs that lead to more stable gradients during training and is less sensitive to outliers in tabular data.

The scaler is fitted using the training subset only and then applied consistently to the validation and test sets. Since the final neural network model is selected using the validation set rather than retrained on the full training data, the test feature matrix can be prepared at this stage. However, the test set remains reserved exclusively for final performance evaluation and is not used during model development.

In [12]:
test_X, test_y = prepare_linear_features(
    df=test_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False,
    reference_columns= train_X.columns.tolist()
)

scaled_test_X = transform_with_scaler(test_X, scaler)

### PyTorch Tensors

Unlike previous models in this project, which operate on pandas DataFrames or NumPy arrays, neural networks implemented in PyTorch work with a specialized data structure called a tensor.

A tensor can be understood as a generalization of a matrix to potentially multiple dimensions. In practice, for tabular data, tensors behave very similarly to standard two-dimensional arrays, where rows represent observations and columns represent features.

The primary difference is that tensors are designed for efficient numerical computation within deep learning frameworks. They support automatic differentiation, which allows PyTorch to compute gradients required for model training, and they are optimized for high-performance operations on both CPUs and GPUs.

In this notebook, the feature matrices and target variables are converted into tensors before training the neural network. While this introduces a new data structure, the underlying representation of the data remains the same as in previous models, ensuring continuity in the overall modeling workflow.

In [13]:
import torch
print(torch.__version__)

2.11.0+cpu


In [14]:

# Features (X)
X_train_tensor = torch.tensor(scaled_train_X.values, dtype=torch.float32)
X_validation_tensor = torch.tensor(scaled_validation_X.values, dtype=torch.float32)
X_test_tensor = torch.tensor(scaled_test_X.values, dtype=torch.float32)

# Targets (y)
y_train_tensor = torch.tensor(train_y.values, dtype=torch.float32).view(-1, 1)
y_validation_tensor = torch.tensor(validation_y.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(test_y.values, dtype=torch.float32).view(-1, 1)

# Quick check
print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_validation_tensor.shape, y_validation_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([4225, 18]) torch.Size([4225, 1])
Validation: torch.Size([1409, 18]) torch.Size([1409, 1])
Test: torch.Size([1409, 18]) torch.Size([1409, 1])


### Mini-Batch Data Loading

After converting the feature matrices and target vectors into PyTorch tensors, the data is wrapped in `TensorDataset` objects and loaded using `DataLoader`.

This step allows the neural network to process the data in mini-batches rather than using the entire dataset at once. Mini-batch training is a standard practice in neural networks because it improves computational efficiency and supports stochastic optimization.

During training, the observations are shuffled at the beginning of each epoch so that the model does not learn from the data in a fixed order. For validation and test evaluation, shuffling is not required, since these datasets are used only for performance assessment.

## Batch Size = 16

In this section, the batch size is fixed at 16, and the remaining hyperparameters are tuned within this setting.

Batch size determines how many observations are processed in a single update step during training. Smaller batch sizes introduce more stochasticity into the optimization process, which can improve generalization but may also lead to noisier training dynamics.

To isolate the effect of batch size, separate models are trained for this configuration, and their performance is evaluated on the validation set. The best-performing model within this batch size setting will later be compared with models trained using alternative batch sizes.

In [21]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
validation_dataset = TensorDataset(X_validation_tensor, y_validation_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training batches: 265
Validation batches: 89
Test batches: 89


### Neural Network Architecture

In [15]:
import torch
import torch.nn as nn

class ChurnNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.0):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

The neural network is implemented as a feed-forward model with a single hidden layer. The input layer corresponds to the set of engineered features, while the hidden layer introduces non-linear transformations through the ReLU activation function.

A dropout layer is optionally included to provide regularization by randomly deactivating a fraction of neurons during training. The output layer consists of a single neuron with a sigmoid activation function, producing a probability estimate for customer churn.

This architecture provides a controlled extension of logistic regression, allowing the model to capture non-linear relationships while maintaining interpretability and simplicity.

### Training Function

In [23]:
from copy import deepcopy
from sklearn.metrics import average_precision_score
import torch
import torch.nn as nn

def train_model(
    input_dim,
    hidden_dim,
    dropout_rate,
    learning_rate,
    weight_decay,
    train_loader,
    validation_loader,
    num_epochs=100,
    patience=10,
    verbose=False
):
    """
    Train a feed-forward neural network and evaluate performance on a validation set.

    This function encapsulates the full training workflow for a single model
    configuration, including model initialization, optimization, and early
    stopping based on validation performance. The model is trained using
    mini-batch gradient descent and evaluated using PR-AUC, which serves as
    the primary model selection metric.

    Parameters
    ----------
    input_dim : int
        Number of input features.
    hidden_dim : int
        Number of neurons in the hidden layer.
    dropout_rate : float
        Dropout rate applied after the hidden layer.
    learning_rate : float
        Learning rate used by the optimizer.
    weight_decay : float
        L2 regularization strength applied through the optimizer.
    train_loader : torch.utils.data.DataLoader
        DataLoader providing mini-batches of training data.
    validation_loader : torch.utils.data.DataLoader
        DataLoader providing mini-batches of validation data.
    num_epochs : int, default=100
        Maximum number of training epochs.
    patience : int, default=10
        Number of consecutive epochs without improvement on validation PR-AUC
        before early stopping is triggered.
    verbose : bool, default=False
        Whether to print epoch-level training progress.

    Returns
    -------
    dict
        Dictionary containing the best fitted model and associated validation
        outputs. Keys include:
        - "model": best model state observed during training
        - "best_val_pr_auc": best validation PR-AUC
        - "best_epoch": epoch at which best validation PR-AUC was observed
        - "val_targets": validation target values
        - "val_probs": predicted validation probabilities
    """

    model = ChurnNN(input_dim, hidden_dim, dropout_rate)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_val_pr_auc = -1.0
    best_epoch = None
    best_model = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        all_probs = []
        all_targets = []

        with torch.no_grad():
            for X_batch, y_batch in validation_loader:
                outputs = model(X_batch)
                all_probs.extend(outputs.detach().cpu().numpy().reshape(-1))
                all_targets.extend(y_batch.detach().cpu().numpy().reshape(-1))

        val_pr_auc = average_precision_score(all_targets, all_probs)

        if verbose:
            print(
                f"Epoch {epoch + 1:03d} | "
                f"Validation PR-AUC: {val_pr_auc:.4f}"
            )

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_epoch = epoch + 1
            best_model = deepcopy(model)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    return {
        "model": best_model,
        "best_val_pr_auc": best_val_pr_auc,
        "best_epoch": best_epoch
    }

To support systematic hyperparameter tuning, the training and evaluation process is encapsulated in a reusable function.

This function performs the full training workflow for a given model configuration. It initializes the neural network, defines the loss function and optimizer, and trains the model using mini-batch gradient descent over multiple epochs.

During training, model performance is evaluated on the validation set after each epoch. The primary evaluation metric is PR-AUC, which reflects the model’s ability to correctly rank customers by churn risk.

Early stopping is used to prevent overfitting. Training is halted when validation performance no longer improves for a specified number of consecutive epochs, ensuring that the model does not continue to fit noise in the training data.

By encapsulating this logic in a function, different hyperparameter configurations can be evaluated consistently and efficiently, enabling a structured comparison of model performance across experiments.

### Hyperparameter Grid

To evaluate different model configurations, a small and controlled hyperparameter grid is defined. Each hyperparameter influences a specific aspect of the model’s behavior, either by controlling model capacity or by affecting the optimization process.

The following hyperparameters are considered:

- **Hidden layer size**  
  This determines the number of neurons in the hidden layer and therefore the capacity of the model. A larger hidden layer allows the network to capture more complex patterns, but also increases the risk of overfitting.

- **Learning rate**  
  The learning rate controls how large each update step is during optimization. Higher values allow the model to learn faster but may lead to unstable training, while lower values result in more gradual updates but may slow convergence.

- **Dropout rate**  
  Dropout is a regularization technique that randomly disables a fraction of neurons during training. For example, a dropout rate of 0.2 means that 20% of neurons are temporarily ignored in each training step, forcing the model to rely on multiple pathways rather than any single feature representation. This helps reduce overfitting and encourages more robust learning.

- **Weight decay**  
  Weight decay applies L2 regularization by penalizing large parameter values. This discourages the model from becoming overly complex and helps improve generalization by keeping the learned weights small.

These hyperparameters are evaluated across a limited set of candidate values to maintain a balance between model flexibility and computational efficiency. Rather than performing an exhaustive search, the focus is on exploring a small number of meaningful configurations.

In [ ]:
hidden_dims = [16, 32, 64]
learning_rates = [0.001, 0.01]
dropout_rates = [0.0, 0.2]
weight_decays = [0.0, 0.001]

### Model Training and Hyperparameter Evaluation

With the model architecture, data loaders, and training function in place, we proceed to fitting the neural network across different hyperparameter configurations.

For the current batch size setting, multiple models are trained using different combinations of hidden layer size, learning rate, dropout rate, and weight decay. Each configuration is evaluated using the training function, which performs mini-batch training over multiple epochs and monitors performance on the validation set.

Model performance is assessed using PR-AUC on the validation data. This metric reflects the model’s ability to correctly rank customers by churn risk and is used as the primary criterion for comparing configurations.

Training is performed iteratively, with early stopping applied to prevent overfitting. If validation performance does not improve for a specified number of epochs, training is halted and the best observed validation score is retained.

The objective of this stage is to identify the best-performing configuration for the current batch size. These results will later be compared across different batch sizes to determine the overall best model.

In [24]:
from src.utils.classification_metrics import compute_classification_metrics
import numpy as np

results = []

for hidden_dim in hidden_dims:
    for learning_rate in learning_rates:
        for dropout_rate in dropout_rates:
            for weight_decay in weight_decays:

                print(
                    f"Training: hidden={hidden_dim}, lr={learning_rate}, "
                    f"dropout={dropout_rate}, wd={weight_decay}"
                )

                # --- Train model ---
                fitted = train_model(
                    input_dim=X_train_tensor.shape[1],
                    hidden_dim=hidden_dim,
                    dropout_rate=dropout_rate,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    train_loader=train_loader,
                    validation_loader=validation_loader,
                    num_epochs=100,
                    patience=10,
                    verbose=False
                )

                best_model = fitted["model"]

                # --- Validation predictions ---
                best_model.eval()
                val_probs = []

                with torch.no_grad():
                    for X_batch, _ in validation_loader:
                        outputs = best_model(X_batch)
                        val_probs.extend(outputs.detach().cpu().numpy().reshape(-1))

                val_probs = np.array(val_probs)
                val_targets = y_validation_tensor.detach().cpu().numpy().reshape(-1)
                val_pred = (val_probs >= 0.5).astype(int)

                # --- Metrics ---
                val_metrics = compute_classification_metrics(
                    y_true=val_targets,
                    y_pred=val_pred,
                    y_prob=val_probs
                )

                # --- Store results ---
                results.append({
                    "batch_size": batch_size,
                    "hidden_dim": hidden_dim,
                    "learning_rate": learning_rate,
                    "dropout_rate": dropout_rate,
                    "weight_decay": weight_decay,
                    "best_epoch": fitted["best_epoch"],
                    "val_pr_auc": fitted["best_val_pr_auc"],
                    "val_auc": val_metrics["auc"],
                    "val_accuracy": val_metrics["accuracy"],
                    "val_precision": val_metrics["precision"],
                    "val_recall": val_metrics["recall"],
                    "val_f1": val_metrics["f1"],
                    "model": best_model
                })

Training: hidden=16, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=64, 

In [25]:
import pandas as pd

results_df = (
    pd.DataFrame(results)
    .drop(columns=["model"])
    .sort_values("val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

results_df.head(10)

,batch_size,hidden_dim,learning_rate,dropout_rate,weight_decay,best_epoch,val_pr_auc,val_auc,val_accuracy,val_precision,val_recall,val_f1
0,16,16,0.010,0.0,0.001,17,0.604618,0.816268,0.782115,0.593838,0.566845,0.580027
1,16,32,0.010,0.2,0.000,11,0.604099,0.815843,0.782115,0.610561,0.494652,0.546529
2,16,64,0.010,0.2,0.000,7,0.603526,0.813138,0.787083,0.624161,0.497326,0.553571
3,16,16,0.001,0.0,0.000,22,0.603273,0.814432,0.781405,0.600000,0.529412,0.562500
4,16,64,0.010,0.0,0.001,6,0.601950,0.813629,0.776437,0.589124,0.521390,0.553191
5,16,64,0.010,0.2,0.001,17,0.601559,0.815111,0.779986,0.628000,0.419786,0.503205
6,16,16,0.001,0.2,0.000,53,0.601306,0.817398,0.785664,0.615385,0.513369,0.559767
7,16,16,0.010,0.2,0.001,11,0.601279,0.816158,0.782115,0.601824,0.529412,0.563300
8,16,16,0.001,0.2,0.001,54,0.600978,0.816739,0.781405,0.612245,0.481283,0.538922
9,16,16,0.010,0.0,0.000,22,0.600277,0.812008,0.778566,0.597484,0.508021,0.549133


Model selection is performed using the validation set, which is also used during training for early stopping. This introduces a mild risk of optimistic bias, as the same data influences both training dynamics and hyperparameter selection.

However, this approach is standard in neural network training, and the presence of a separate hold-out test set ensures that final performance evaluation remains unbiased.

The top-performing model is selected based on validation PR-AUC. While several configurations achieve similar performance, the best model also maintains a balanced trade-off between precision and recall, resulting in the highest F1 score among the top candidates. No alternative configuration provides a clear improvement across other metrics without sacrificing PR-AUC.

In [27]:
best_result_batch_16 = max(results, key=lambda x: x["val_pr_auc"])
best_result_batch_16

{'batch_size': 16,
 'hidden_dim': 16,
 'learning_rate': 0.01,
 'dropout_rate': 0.0,
 'weight_decay': 0.001,
 'best_epoch': 17,
 'val_pr_auc': 0.6046175001652246,
 'val_auc': 0.816267534681857,
 'val_accuracy': np.float64(0.7821149751596878),
 'val_precision': np.float64(0.5938375350140056),
 'val_recall': np.float64(0.5668449197860963),
 'val_f1': np.float64(0.5800273597811217),
 'model': ChurnNN(
   (model): Sequential(
     (0): Linear(in_features=18, out_features=16, bias=True)
     (1): ReLU()
     (2): Dropout(p=0.0, inplace=False)
     (3): Linear(in_features=16, out_features=1, bias=True)
     (4): Sigmoid()
   )
 )}

To reduce the risk of overfitting to the validation set, the hyperparameter search is not further refined. Unlike earlier models, where additional tuning stages were explored, the neural network already exhibits a clear performance plateau, with all tested configurations achieving very similar PR-AUC values.

Given the limited variation in performance across configurations, further expanding or refining the hyperparameter grid is unlikely to yield meaningful improvements and may instead introduce additional selection bias.

## Batch Size = 32

In this section, the batch size is fixed at 32, and the remaining hyperparameters are tuned within this setting.

In [28]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
validation_dataset = TensorDataset(X_validation_tensor, y_validation_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training batches: 133
Validation batches: 45
Test batches: 45


Training Function and Model Architecture can be reused.

### Hyperparameter Grid

Same as with the batch 16.

In [29]:
hidden_dims = [16, 32, 64]
learning_rates = [0.001, 0.01]
dropout_rates = [0.0, 0.2]
weight_decays = [0.0, 0.001]

### Model Fitting and Evaluation

In [30]:
from src.utils.classification_metrics import compute_classification_metrics
import numpy as np

results = []

for hidden_dim in hidden_dims:
    for learning_rate in learning_rates:
        for dropout_rate in dropout_rates:
            for weight_decay in weight_decays:

                print(
                    f"Training: hidden={hidden_dim}, lr={learning_rate}, "
                    f"dropout={dropout_rate}, wd={weight_decay}"
                )

                # --- Train model ---
                fitted = train_model(
                    input_dim=X_train_tensor.shape[1],
                    hidden_dim=hidden_dim,
                    dropout_rate=dropout_rate,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    train_loader=train_loader,
                    validation_loader=validation_loader,
                    num_epochs=100,
                    patience=10,
                    verbose=False
                )

                best_model = fitted["model"]

                # --- Validation predictions ---
                best_model.eval()
                val_probs = []

                with torch.no_grad():
                    for X_batch, _ in validation_loader:
                        outputs = best_model(X_batch)
                        val_probs.extend(outputs.detach().cpu().numpy().reshape(-1))

                val_probs = np.array(val_probs)
                val_targets = y_validation_tensor.detach().cpu().numpy().reshape(-1)
                val_pred = (val_probs >= 0.5).astype(int)

                # --- Metrics ---
                val_metrics = compute_classification_metrics(
                    y_true=val_targets,
                    y_pred=val_pred,
                    y_prob=val_probs
                )

                # --- Store results ---
                results.append({
                    "batch_size": batch_size,
                    "hidden_dim": hidden_dim,
                    "learning_rate": learning_rate,
                    "dropout_rate": dropout_rate,
                    "weight_decay": weight_decay,
                    "best_epoch": fitted["best_epoch"],
                    "val_pr_auc": fitted["best_val_pr_auc"],
                    "val_auc": val_metrics["auc"],
                    "val_accuracy": val_metrics["accuracy"],
                    "val_precision": val_metrics["precision"],
                    "val_recall": val_metrics["recall"],
                    "val_f1": val_metrics["f1"],
                    "model": best_model
                })

Training: hidden=16, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=64, 

In [31]:
import pandas as pd

results_df = (
    pd.DataFrame(results)
    .drop(columns=["model"])
    .sort_values("val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

results_df.head(10)

,batch_size,hidden_dim,learning_rate,dropout_rate,weight_decay,best_epoch,val_pr_auc,val_auc,val_accuracy,val_precision,val_recall,val_f1
0,32,16,0.010,0.2,0.001,12,0.602754,0.815442,0.780696,0.594752,0.545455,0.569038
1,32,16,0.010,0.2,0.000,7,0.601230,0.815173,0.777857,0.592705,0.521390,0.554765
2,32,16,0.001,0.2,0.001,46,0.600824,0.815724,0.789212,0.627063,0.508021,0.561300
3,32,64,0.010,0.0,0.001,21,0.600388,0.813450,0.783534,0.610932,0.508021,0.554745
4,32,32,0.010,0.0,0.000,5,0.600311,0.815907,0.775727,0.587349,0.521390,0.552408
5,32,32,0.010,0.2,0.001,18,0.600238,0.816974,0.778566,0.607639,0.467914,0.528701
6,32,16,0.010,0.0,0.000,21,0.600167,0.812616,0.782825,0.621429,0.465241,0.532110
7,32,64,0.010,0.2,0.001,20,0.599702,0.814460,0.783534,0.641975,0.417112,0.505673
8,32,64,0.001,0.2,0.000,16,0.599463,0.815295,0.777147,0.598039,0.489305,0.538235
9,32,32,0.001,0.2,0.000,23,0.598978,0.814003,0.782825,0.605590,0.521390,0.560345


As with the previous batch size, model performance across configurations is highly consistent, with only minor variation in PR-AUC. The top-performing model is selected based on validation PR-AUC, and no alternative configuration demonstrates a clear improvement across other metrics without sacrificing this criterion.

This further supports the observation of a performance plateau, suggesting that additional hyperparameter refinement is unlikely to yield substantial gains.

In [32]:
best_result_batch_32 = max(results, key=lambda x: x["val_pr_auc"])
best_result_batch_32

{'batch_size': 32,
 'hidden_dim': 16,
 'learning_rate': 0.01,
 'dropout_rate': 0.2,
 'weight_decay': 0.001,
 'best_epoch': 12,
 'val_pr_auc': 0.6027539944941076,
 'val_auc': 0.8154421452375417,
 'val_accuracy': np.float64(0.7806955287437899),
 'val_precision': np.float64(0.5947521865889213),
 'val_recall': np.float64(0.5454545454545454),
 'val_f1': np.float64(0.5690376569037656),
 'model': ChurnNN(
   (model): Sequential(
     (0): Linear(in_features=18, out_features=16, bias=True)
     (1): ReLU()
     (2): Dropout(p=0.2, inplace=False)
     (3): Linear(in_features=16, out_features=1, bias=True)
     (4): Sigmoid()
   )
 )}

## Batch Size = 64

In this section, the batch size is fixed at 64, and the remaining hyperparameters are tuned within this setting.

In [33]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
validation_dataset = TensorDataset(X_validation_tensor, y_validation_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training batches: 67
Validation batches: 23
Test batches: 23


Training Function and Model Architecture can be reused.

### Hyperparameter Grid

Same as with the batches 16 and 32.

In [34]:
hidden_dims = [16, 32, 64]
learning_rates = [0.001, 0.01]
dropout_rates = [0.0, 0.2]
weight_decays = [0.0, 0.001]

### Model Fitting and Evaluation

In [35]:
from src.utils.classification_metrics import compute_classification_metrics
import numpy as np

results = []

for hidden_dim in hidden_dims:
    for learning_rate in learning_rates:
        for dropout_rate in dropout_rates:
            for weight_decay in weight_decays:

                print(
                    f"Training: hidden={hidden_dim}, lr={learning_rate}, "
                    f"dropout={dropout_rate}, wd={weight_decay}"
                )

                # --- Train model ---
                fitted = train_model(
                    input_dim=X_train_tensor.shape[1],
                    hidden_dim=hidden_dim,
                    dropout_rate=dropout_rate,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    train_loader=train_loader,
                    validation_loader=validation_loader,
                    num_epochs=100,
                    patience=10,
                    verbose=False
                )

                best_model = fitted["model"]

                # --- Validation predictions ---
                best_model.eval()
                val_probs = []

                with torch.no_grad():
                    for X_batch, _ in validation_loader:
                        outputs = best_model(X_batch)
                        val_probs.extend(outputs.detach().cpu().numpy().reshape(-1))

                val_probs = np.array(val_probs)
                val_targets = y_validation_tensor.detach().cpu().numpy().reshape(-1)
                val_pred = (val_probs >= 0.5).astype(int)

                # --- Metrics ---
                val_metrics = compute_classification_metrics(
                    y_true=val_targets,
                    y_pred=val_pred,
                    y_prob=val_probs
                )

                # --- Store results ---
                results.append({
                    "batch_size": batch_size,
                    "hidden_dim": hidden_dim,
                    "learning_rate": learning_rate,
                    "dropout_rate": dropout_rate,
                    "weight_decay": weight_decay,
                    "best_epoch": fitted["best_epoch"],
                    "val_pr_auc": fitted["best_val_pr_auc"],
                    "val_auc": val_metrics["auc"],
                    "val_accuracy": val_metrics["accuracy"],
                    "val_precision": val_metrics["precision"],
                    "val_recall": val_metrics["recall"],
                    "val_f1": val_metrics["f1"],
                    "model": best_model
                })

Training: hidden=16, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=16, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.001, dropout=0.2, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.0, wd=0.001
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.0
Training: hidden=32, lr=0.01, dropout=0.2, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.0
Training: hidden=64, lr=0.001, dropout=0.0, wd=0.001
Training: hidden=64, lr=0.001, dropout=0.2, wd=0.0
Training: hidden=64, 

In [36]:
import pandas as pd

results_df = (
    pd.DataFrame(results)
    .drop(columns=["model"])
    .sort_values("val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

results_df.head(10)

,batch_size,hidden_dim,learning_rate,dropout_rate,weight_decay,best_epoch,val_pr_auc,val_auc,val_accuracy,val_precision,val_recall,val_f1
0,64,64,0.010,0.2,0.001,12,0.605908,0.813487,0.782115,0.623616,0.451872,0.524031
1,64,64,0.010,0.0,0.001,35,0.605608,0.815755,0.777147,0.584746,0.553476,0.568681
2,64,32,0.001,0.2,0.001,18,0.600641,0.812949,0.780696,0.605178,0.500000,0.547584
3,64,16,0.010,0.0,0.000,10,0.600036,0.813768,0.779986,0.623077,0.433155,0.511041
4,64,16,0.010,0.2,0.001,7,0.599863,0.814264,0.776437,0.594855,0.494652,0.540146
5,64,64,0.010,0.0,0.000,12,0.599739,0.814956,0.780696,0.605178,0.500000,0.547584
6,64,32,0.001,0.0,0.001,28,0.599057,0.814654,0.777147,0.597403,0.491979,0.539589
7,64,16,0.010,0.0,0.001,16,0.598507,0.813342,0.782115,0.600601,0.534759,0.565771
8,64,32,0.010,0.0,0.000,8,0.598005,0.813179,0.781405,0.613014,0.478610,0.537538
9,64,64,0.001,0.2,0.000,35,0.597879,0.812817,0.781405,0.604430,0.510695,0.553623


Although the top configuration achieves the highest PR-AUC, the difference compared to the second-best model is negligible. The second configuration provides a substantially better balance between precision and recall, resulting in a higher F1 score.

Given the objective of identifying customers at risk of churn, improved recall is particularly valuable. Therefore, the second configuration is selected as the preferred model for this batch size.

In [37]:
best_result_batch_64 = next(
    r for r in results
    if (
        r["batch_size"] == 64 and
        r["hidden_dim"] == 64 and
        r["learning_rate"] == 0.01 and
        r["dropout_rate"] == 0.0 and
        r["weight_decay"] == 0.001
    )
)

In [38]:
best_result_batch_64

{'batch_size': 64,
 'hidden_dim': 64,
 'learning_rate': 0.01,
 'dropout_rate': 0.0,
 'weight_decay': 0.001,
 'best_epoch': 35,
 'val_pr_auc': 0.605607758577543,
 'val_auc': 0.815754734041179,
 'val_accuracy': np.float64(0.7771469127040455),
 'val_precision': np.float64(0.5847457627118644),
 'val_recall': np.float64(0.553475935828877),
 'val_f1': np.float64(0.5686813186813187),
 'model': ChurnNN(
   (model): Sequential(
     (0): Linear(in_features=18, out_features=64, bias=True)
     (1): ReLU()
     (2): Dropout(p=0.0, inplace=False)
     (3): Linear(in_features=64, out_features=1, bias=True)
     (4): Sigmoid()
   )
 )}

In [39]:
import pickle

reference_columns = train_X.columns.tolist()

nn_artifact_batch_64 = {
    "model_type": "pytorch_nn",
    
    # model
    "model": best_result_batch_64["model"].eval(),
    
    # preprocessing config
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": reference_columns,
    
    # scaler
    "scaler": scaler,
    
    # model params
    "params": {
        "batch_size": best_result_batch_64["batch_size"],
        "hidden_dim": best_result_batch_64["hidden_dim"],
        "learning_rate": best_result_batch_64["learning_rate"],
        "dropout_rate": best_result_batch_64["dropout_rate"],
        "weight_decay": best_result_batch_64["weight_decay"]
    },
    ## nn specific
    "input_dim": X_train_tensor.shape[1]
}

nn_artifact_batch_32 = {
    "model_type": "pytorch_nn",
    
    # model
    "model": best_result_batch_32["model"].eval(),
    
    # preprocessing config
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": reference_columns,
    
    # scaler
    "scaler": scaler,
    
    # model params
    "params": {
        "batch_size": best_result_batch_32["batch_size"],
        "hidden_dim": best_result_batch_32["hidden_dim"],
        "learning_rate": best_result_batch_32["learning_rate"],
        "dropout_rate": best_result_batch_32["dropout_rate"],
        "weight_decay": best_result_batch_32["weight_decay"]
    },
    ## nn specific
    "input_dim": X_train_tensor.shape[1]
}

nn_artifact_batch_16 = {
    "model_type": "pytorch_nn",
    
    # model
    "model": best_result_batch_16["model"].eval(),
    
    # preprocessing config
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": reference_columns,
    
    # scaler
    "scaler": scaler,
    
    # model params
    "params": {
        "batch_size": best_result_batch_16["batch_size"],
        "hidden_dim": best_result_batch_16["hidden_dim"],
        "learning_rate": best_result_batch_16["learning_rate"],
        "dropout_rate": best_result_batch_16["dropout_rate"],
        "weight_decay": best_result_batch_16["weight_decay"]
    },
    ## nn specific
    "input_dim": X_train_tensor.shape[1]
}


with open("./models/nn_64.pkl", "wb") as f:
    pickle.dump(nn_artifact_batch_64, f)

with open("./models/nn_32.pkl", "wb") as f:
    pickle.dump(nn_artifact_batch_32, f)

with open("./models/nn_16.pkl", "wb") as f:
    pickle.dump(nn_artifact_batch_16, f)

With the best-performing configurations identified and saved, the final step is to evaluate the selected models on the hold-out test set to obtain an unbiased estimate of performance.

### Feature Importance and Interpretability

Unlike logistic regression and tree-based models, neural networks do not provide a straightforward notion of feature importance. Although the model is parameterized through learned weights, these weights are distributed across multiple layers and non-linear transformations, making direct interpretation difficult.

For this reason, feature importance is not analyzed in the same form as in earlier models. Instead, the neural network is treated primarily as a predictive benchmark, while interpretability remains grounded in the more transparent models developed in previous stages of the project.

This reflects an important trade-off in model selection. Neural networks can offer greater flexibility and improved predictive performance, but often at the cost of reduced transparency and interpretability.

## Test Set Evaluation

### Test Set Evaluation

With the best-performing configuration identified for each batch size, the selected models can now be evaluated on the hold-out test set.

This step provides an unbiased estimate of model performance, as the test data has not been used during training, early stopping, or hyperparameter selection. Unlike the validation set, which guides model development, the test set serves only as a final assessment of how well the selected models generalize to unseen data.

For each batch size, the corresponding best model is used to generate predicted probabilities on the test set. These probabilities are then converted to class predictions using the default threshold of 0.5, and the same evaluation framework used throughout the project is applied.

The objective of this stage is twofold. First, it allows each selected neural network configuration to be assessed under identical conditions on unseen data. Second, it provides the basis for comparing the final batch-size-specific models and selecting the overall preferred neural network specification.

In [40]:
import numpy as np
import pandas as pd
import torch

def evaluate_model_on_test(model, test_loader, y_test_tensor):
    """
    Evaluate a trained neural network on the test set.

    Parameters
    ----------
    model : torch.nn.Module
        Trained neural network model.
    test_loader : torch.utils.data.DataLoader
        DataLoader providing mini-batches of test data.
    y_test_tensor : torch.Tensor
        True test target values.

    Returns
    -------
    dict
        Dictionary containing classification metrics computed on the test set.
    """
    model.eval()

    test_probs = []

    with torch.no_grad():
        for X_batch, _ in test_loader:
            outputs = model(X_batch)
            test_probs.extend(outputs.detach().cpu().numpy().reshape(-1))

    test_probs = np.array(test_probs)
    test_targets = y_test_tensor.detach().cpu().numpy().reshape(-1)
    test_pred = (test_probs >= 0.5).astype(int)

    test_metrics = compute_classification_metrics(
        y_true=test_targets,
        y_pred=test_pred,
        y_prob=test_probs
    )

    return test_metrics

In [41]:
test_results = []

selected_results = [
    best_result_batch_16,
    best_result_batch_32,
    best_result_batch_64
]

for result in selected_results:
    batch_size = result["batch_size"]
    model = result["model"]

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_metrics = evaluate_model_on_test(
        model=model,
        test_loader=test_loader,
        y_test_tensor=y_test_tensor
    )

    test_results.append({
        "batch_size": result["batch_size"],
        "hidden_dim": result["hidden_dim"],
        "learning_rate": result["learning_rate"],
        "dropout_rate": result["dropout_rate"],
        "weight_decay": result["weight_decay"],
        "best_epoch": result["best_epoch"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_auc": test_metrics["auc"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"]
    })

test_results_df = (
    pd.DataFrame(test_results)
    .sort_values("test_pr_auc", ascending=False)
    .reset_index(drop=True)
)

test_results_df

,batch_size,hidden_dim,learning_rate,dropout_rate,weight_decay,best_epoch,test_pr_auc,test_auc,test_accuracy,test_precision,test_recall,test_f1
0,32,16,0.01,0.2,0.001,12,0.659241,0.836637,0.794180,0.631250,0.540107,0.582133
1,16,16,0.01,0.0,0.001,17,0.654900,0.834364,0.786373,0.607670,0.550802,0.577840
2,64,64,0.01,0.0,0.001,35,0.649906,0.833563,0.788502,0.612426,0.553476,0.581461


### Comparison of Batch-Size-Specific Neural Network Models

The final comparison across batch sizes shows that the model trained with a batch size of 32 achieves the strongest performance on the test set, as measured by PR-AUC.

Among the three batch-size-specific configurations, this model provides the best overall balance of predictive ranking performance and classification metrics. In particular, it achieves the highest test PR-AUC while also maintaining a strong balance between precision and recall.

The results suggest that batch size has a meaningful influence on optimization behavior and generalization. In this setting, a batch size of 32 appears to provide the most effective trade-off between stochasticity and stability during training.

The best-performing neural network within this family uses a relatively small hidden layer, indicating that a modest level of model complexity is sufficient to capture the relevant non-linear structure in the data.

At this stage, the objective is not to select the final model for the project as a whole, but rather to identify the strongest neural-network candidate for subsequent comparison against the other modeling approaches developed earlier.

## Network Executive Summary

This section introduced a neural network model as a flexible, non-linear extension of the previously explored approaches. The objective was to assess whether a modest increase in model complexity could improve predictive performance while maintaining a controlled and interpretable modeling framework.

A shallow feed-forward architecture with a single hidden layer was used, providing a direct extension of logistic regression. This design allows the model to capture non-linear relationships without introducing the additional complexity associated with deeper networks, which are typically less effective for structured tabular data.

Model training was performed using mini-batch gradient descent with the Adam optimizer. Key hyperparameters, including hidden layer size, learning rate, dropout rate, weight decay, and batch size, were explored through a structured but intentionally limited search. Early stopping was applied to prevent overfitting, and model comparison was based on PR-AUC, reflecting the importance of correctly identifying churners in an imbalanced setting.

The results showed that neural network performance was relatively stable across hyperparameter configurations, with only minor variation in validation PR-AUC. This indicates a performance plateau, suggesting that further hyperparameter refinement would likely yield limited gains while increasing the risk of overfitting to the validation set.

Batch size was found to have a meaningful impact on performance. A batch size of 32 provided the best balance between stochasticity and stability during optimization, leading to the strongest generalization performance on the test set. In contrast, smaller batches introduced more noise, while larger batches resulted in less effective generalization.

The best-performing neural network used a relatively small hidden layer, indicating that the dataset does not require high model capacity to capture relevant patterns. This reinforces the importance of controlled model complexity when working with tabular data.

From an interpretability perspective, the neural network does not provide a direct measure of feature importance, highlighting a trade-off between predictive flexibility and transparency. As a result, the model is treated primarily as a predictive benchmark rather than an interpretable solution.

Overall, the neural network achieved competitive performance while maintaining a simple and well-regularized structure. The best configuration identified in this section will be carried forward as the representative neural network model for comparison with other modeling approaches.